In [1]:
from google.colab import drive
from sqlalchemy import create_engine
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
import plotly.express as px
import glob
import os

In [2]:
# Montar Google Drive
drive.mount('/content/drive')

# Definir rutas de archivos y conexión
ruta_base = "/content/drive/My Drive/Colab Notebooks/Curso ETL/Proyecto_ETL_Comercio/Data/"

Mounted at /content/drive


In [3]:
# EXTRACT (extracción de datos)
# Dataset DANE EMICRON
def extract_emicron():
    df = pd.read_csv(ruta_base + "EMICRON.csv")
    return df

# Datos DIVIPOLA
def extract_divipola():
    df = pd.read_csv(ruta_base + "DIVIPOLA.csv")
    df = df[[
        "Código Departamento",
        "Nombre Departamento"
    ]].drop_duplicates()
    df["Nombre Departamento"] = df["Nombre Departamento"].str.title()
    df.columns = ["cod_dep", "departamento"]
    return df

# Dataset CIIU
def extract_ciiu():
    df = pd.read_csv(ruta_base + "CIIU.csv")
    df.columns = df.columns.str.strip().str.lower()
    # columna original
    col = "clasificacion ciiu"
    # extraer código tipo A0112 → 0112
    df["ciiu"] = df[col].str.extract(r'([A-Z]?)(\d{2,4})')[1]
    # descripción limpia
    df["descripcion"] = df[col].str.replace(r'^[A-Z]?\d+\s*\*\*\s*', '', regex=True)
    return df[["ciiu", "descripcion"]]

# Dataset IPC
def extract_ipc():
    df = pd.read_excel(ruta_base + "IPC.xlsx")
    df.columns = df.columns.str.strip().str.lower()
    df = df.rename(columns={
        "año": "anio",
        "numero indice": "ipc"})
    mes_map = {
    "enero": 1, "febrero": 2, "marzo": 3,
    "abril": 4, "mayo": 5, "junio": 6,
    "julio": 7, "agosto": 8, "septiembre": 9,
    "octubre": 10, "noviembre": 11, "diciembre": 12}
    df["mes"] = df["mes"].str.lower().map(mes_map)
    return df[["anio", "mes", "ipc"]]

# Dataset PIB
def extract_pib():
    df = pd.read_excel(ruta_base + "PIB.xlsx")
    df = df.melt(
        id_vars=["Código Departamento (DIVIPOLA)", "DEPARTAMENTOS"],
        var_name="anio",
        value_name="pib")
    df = df.rename(columns={
        "Código Departamento (DIVIPOLA)": "cod_dep",
        "DEPARTAMENTOS": "departamento"})
    # limpiar años (2023p, 2024pr)
    df["anio"] = df["anio"].astype(str).str.extract(r'(\d+)').astype(int)
    return df

# Dataset EMPRESAS
def extract_empresas():
    df = pd.read_csv(ruta_base + "EMPRESAS.csv")#, encoding="latin1")
    df.columns = df.columns.str.strip().str.lower()
    df = df.rename(columns={
        #"cIIu-1".lower(): "ciiu",
        "tam-empresa".lower(): "tam_empresa",
        "personal": "personal"})
    # columna original
    col = "ciiu-1"
    # extraer código tipo A0112 → 0112
    df["ciiu"] = df[col].str.extract(r'([A-Z]?)(\d{2,4})')[1]
    # descripción limpia
    df["descripcion"] = df[col].str.replace(r'^[A-Z]?\d+\s*\*\*\s*', '', regex=True)
    return df[["ciiu", "descripcion", "tam_empresa", "personal"]]

In [4]:
#Ejecucion de extraccion datasets
emicron = extract_emicron()
divipola = extract_divipola()
ciiu = extract_ciiu()
ipc = extract_ipc()
pib = extract_pib()
empresas = extract_empresas()

In [5]:
print("EMICRON:\n", emicron.head())
print("EMICRON:\n", emicron.info())
print("DIVIPOLA:\n", divipola.head())
print("DIVIPOLA:\n", divipola.info())
print("CIIU:\n", ciiu.head())
print("CIIU:\n", ciiu.info())
print("IPC:\n", ipc.head())
print("IPC:\n", ipc.info())
print("PIB:\n", pib.head())
print("PIB:\n", pib.info())
print("EMPRESAS:\n", empresas.head())
print("EMPRESAS:\n", empresas.info())

EMICRON:
    DIRECTORIO  SECUENCIA_P  SECUENCIA_ENCUESTA     P3057  P3058  P3059  P3060  \
0     7627444            1                   1  300000.0    0.0    0.0    0.0   
1     7627446            1                   1       NaN    NaN    NaN    NaN   
2     7627449            1                   1       NaN    NaN    NaN    NaN   
3     7627453            1                   2       NaN    NaN    NaN    NaN   
4     7627456            1                   3       NaN    NaN    NaN    NaN   

      P3061  P3062  P4002  ...   P3072  VENTAS_MES_ANTERIOR  \
0       NaN    NaN    NaN  ...  300000               300000   
1       NaN    NaN    NaN  ...  800000               940000   
2  300000.0    0.0    0.0  ...  150000               300000   
3       NaN    NaN    NaN  ...  650000              1500000   
4       NaN    NaN    NaN  ...  500000              1200000   

   VENTAS_MES_ANIO_ANTERIOR  VENTAS_ANIO_ANTERIOR  VALOR_AGREGADO  \
0                  280000.0             3000000.0      

In [6]:
# EDA
# ANALISIS GENERAL
def eda_general(df, nombre="dataset"):
    print(f"\n EDA - {nombre}")
    print("-" * 50)
    print("Shape:", df.shape)
    print("\nTipos de datos:\n", df.dtypes)
    print("\nValores nulos:\n", df.isna().sum())
    print("\nEstadísticos:\n", df.describe())

# DISTRIBUCIÓN DE VARIABLES CLAVE
def eda_ventas(df):

    fig = px.histogram(df, x="VENTAS_MES_ANTERIOR",
                       title="Distribución de Ventas")
    fig.show()

    df["log_ventas"] = np.log1p(df["VENTAS_MES_ANTERIOR"])
    fig = px.histogram(df, x="log_ventas",
                      title="Distribución Logarítmica de Ventas")
    fig.show()

    fig = px.box(df, y="VENTAS_MES_ANTERIOR",
                 title="Outliers en Ventas")
    fig.show()

    ventas_dep = df.groupby("COD_DEPTO")["VENTAS_MES_ANTERIOR"].sum().reset_index()

    fig = px.bar(ventas_dep,
                x="COD_DEPTO",
                y="VENTAS_MES_ANTERIOR",
                title="Ventas Totales por Departamento")
    fig.show()

# DETECCION DE OUTLIERS
def detectar_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"Outliers en {col}: {len(outliers)}")
    return outliers

# VALIDACION DE JOINS
def validar_merge(left, right, key):
    print(f"\n Validando merge por {key}")
    left_keys = set(left[key].astype(str))
    right_keys = set(right[key].astype(str))
    print("Valores en left no en right:", len(left_keys - right_keys))
    print("Valores en right no en left:", len(right_keys - left_keys))

# CONSISTENCIA DE VARIABLES
def validar_negativos(df):
    cols = ["VENTAS_MES_ANTERIOR", "VALOR_AGREGADO"]

    for col in cols:
        negativos = (df[col] < 0).sum()
        print(f"{col} negativos:", negativos)

In [7]:
# EJECUCION DEL EDA
eda_general(emicron, "EMICRON")
eda_general(ipc, "IPC")
eda_general(pib, "PIB")
eda_ventas(emicron)
detectar_outliers(emicron, "VENTAS_MES_ANTERIOR")
validar_negativos(emicron)


 EDA - EMICRON
--------------------------------------------------
Shape: (78501, 72)

Tipos de datos:
 DIRECTORIO              int64
SECUENCIA_P             int64
SECUENCIA_ENCUESTA      int64
P3057                 float64
P3058                 float64
                       ...   
INGRESO_MIXTO           int64
CLASE_TE                int64
COD_DEPTO               int64
AREA                  float64
F_EXP                 float64
Length: 72, dtype: object

Valores nulos:
 DIRECTORIO                0
SECUENCIA_P               0
SECUENCIA_ENCUESTA        0
P3057                 71552
P3058                 71552
                      ...  
INGRESO_MIXTO             0
CLASE_TE                  0
COD_DEPTO                 0
AREA                  26071
F_EXP                     0
Length: 72, dtype: int64

Estadísticos:
          DIRECTORIO   SECUENCIA_P  SECUENCIA_ENCUESTA         P3057  \
count  7.850100e+04  78501.000000        78501.000000  6.949000e+03   
mean   7.820956e+06      1.00777

Outliers en VENTAS_MES_ANTERIOR: 7020
VENTAS_MES_ANTERIOR negativos: 0
VALOR_AGREGADO negativos: 2665


In [10]:
# TRANSFORM (transformación)
def transform_data(emicron, divipola, ciiu, ipc, pib):
    df = emicron[[
        "COD_DEPTO",
        "P3061",
        "VENTAS_MES_ANTERIOR",
        "VENTAS_ANIO_ANTERIOR",
        "VALOR_AGREGADO",
        "INGRESO_MIXTO",
        "CLASE_TE"
    ]].copy()

    df.columns = [
        "cod_dep",
        "ciiu",
        "ventas_mes",
        "ventas_anio",
        "valor_agregado",
        "ingreso_mixto",
        "tipo_empresa"]

    # LIMPIEZA
    df = df.dropna()
    df = df[df["ventas_mes"] > 0]

    # eliminar outliers extremos
    df = df[df["ventas_mes"] < df["ventas_mes"].quantile(0.99)]

    # JOINS
    df = df.merge(divipola, on="cod_dep", how="left")

    # FECHA
    df["anio"] = 2024
    df["mes"] = 1
    df = df.merge(ipc, on=["anio", "mes"], how="left")
    df = df.merge(pib, on=["cod_dep", "anio"], how="left")

    # KPIs
    df["total_venta"] = df["ventas_mes"]
    df["venta_real"] = df["total_venta"] / df["ipc"]
    df["crecimiento"] = df["ventas_mes"] / (df["ventas_anio"] + 1)
    df["productividad"] = df["valor_agregado"] / df["ventas_mes"]
    df["intensidad_economica"] = df["total_venta"] / df["pib"]

    return df

In [11]:
df_final = transform_data(emicron, divipola, ciiu, ipc, pib)
df_final.head()
eda_general(df_final, "DATASET_FINAL")


 EDA - DATASET_FINAL
--------------------------------------------------
Shape: (16860, 18)

Tipos de datos:
 cod_dep                   int64
ciiu                    float64
ventas_mes                int64
ventas_anio             float64
valor_agregado            int64
ingreso_mixto             int64
tipo_empresa              int64
departamento_x           object
anio                      int64
mes                       int64
ipc                     float64
departamento_y           object
pib                     float64
total_venta               int64
venta_real              float64
crecimiento             float64
productividad           float64
intensidad_economica    float64
dtype: object

Valores nulos:
 cod_dep                 0
ciiu                    0
ventas_mes              0
ventas_anio             0
valor_agregado          0
ingreso_mixto           0
tipo_empresa            0
departamento_x          0
anio                    0
mes                     0
ipc                    

In [12]:
#Grafica Ventas por departamento
fig = px.bar(df_final, x="departamento_x", y="total_venta",
             title="Ventas por Departamento")
fig.show()

#Grafica Productividad
fig = px.histogram(df_final, x="productividad",
                   title="Distribución de Productividad")
fig.show()

#Grafica Ventas Vs PIB
fig = px.scatter(df_final,
                 x="pib",
                 y="total_venta",
                 color="cod_dep",
                 title="Ventas vs PIB")
fig.show()


In [13]:
# Crear carpeta si no existe
os.makedirs(ruta_base, exist_ok=True)

# Guardar CSV
df_final.to_csv(ruta_base + "dataset_final.csv", index=False)

#Guardar XLSX para BI
df_final.to_excel(ruta_base + "dataset_final.xlsx", index=False)

print("Dataset guardado correctamente en Drive")

Dataset guardado correctamente en Drive
